In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import os
import psutil, gc
import matplotlib as mpl

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_04242026.h5ad")

## subset the adata to only include cell types of interest
adata = adata[adata.obs['Cell_class'].isin(['Astrocyte'])].copy()

print(adata)
print(adata.X[:10,:10])

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 

print(adata)
print(adata.X[:10,:10])

### Performing another round of integration using Harmony

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='individualID', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=0.6,key_added='leiden_harmony')

In [ ]:
# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony",'region_layer'],
    # color=["leiden_harmony", "individualID",'brain_region','Cell_type'],
    frameon=False,
    ncols=3,
    size=20,
    # legend_loc="on data",
    # save=f"_Astrocyte_umap_harmony.svg"
)

In [ ]:
### Given some known cell type markers, annotate the clusters
adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)

markers = ['AQP4','GFAP','RFX4','PITPNC1','FAM189A2','BMPR1B','GLIS3','GPC5','CD44','ADGRV1','SLC1A3','RYR3','WDR49','PARD3B','SORBS1','RORA']
# markers = {'1.1':"KCNT2",'1.2':"LAMA2",'1.3':'EPS8','2.1':"SLC4A4",'2.2':"SLIT2",'2.3':'SLC26A2','3.1':"KCNIP1",'3.2':"RNF220",'3.3':"LAMC3",'4.1':"CARMN",'4.2':"RGS6",'4.3':'PRKG1','5.1':'TRPM3','5.2':'SLC13A3'}
sc.pl.dotplot(adata, markers, groupby='leiden_harmony',use_raw=False)#, save=f"_Astrocyte_markers_dotplot.svg")

In [ ]:
### wilcox test to find the top markers for each cluster
sc.tl.rank_genes_groups(adata, groupby='leiden_harmony', method='wilcoxon', key_added='rank_genes_harmony')
### show the top 10 markers for each cluster
result = adata.uns['rank_genes_harmony']
groups = result['names'].dtype.names
top_markers = pd.DataFrame({group: result['names'][group][:25] for group in groups})
print(top_markers)

In [ ]:
# sc.tl.dendrogram(adata,groupby="leiden_harmony")
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    key='rank_genes_harmony',
    groupby='leiden_harmony',
    standard_scale='var',      # 'var' (per gene) or 'obs' (per cell)
    var_group_rotation=45,     # rotate gene labels
    colorbar_title='Mean\nexpression',
    size_title='Fraction\nexpressing',
    figsize=(15, 7),
    # save='_marker_dotplot.pdf'  # optional save
)

In [ ]:
## make the results of the wilcox test into a dataframe including the p value, adjusted p-values and log fold changes
wilcox_results = []
for cluster in groups:
    for gene, pval, pval_adj, logfc in zip(result['names'][cluster], result['pvals'][cluster], result['pvals_adj'][cluster], result['logfoldchanges'][cluster]):
        wilcox_results.append({'cluster': cluster, 'gene': gene, 'pval': pval, 'pval_adj': pval_adj, 'logfc': logfc})
wilcox_df = pd.DataFrame(wilcox_results)
display(wilcox_df.head())

## showing the top 20 markers with smallest adjusted p-values in dataframe for each cluster and the log fold changes should be greater than 0.5
top20_markers_df = wilcox_df[wilcox_df['logfc'] > 0.5].groupby('cluster').apply(lambda x: x.nsmallest(20, 'pval_adj')).reset_index(drop=True)
display(top20_markers_df)

# top20_markers_df.to_csv(f"{PATH}/Results/Revision_2/DEG/Astrocyte_top20_markers_harmony.csv", index=False)

In [ ]:
sc.pl.rank_genes_groups(adata, groupby='leiden_harmony', key='rank_genes_harmony', n_genes=20, use_raw=False) #, save=f"_Fibroblast_top20_markers_dotplot.svg")

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()

print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Astrocyte_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)

## Builting region-level representation (Donor-balanced centroids in PC spave)

In [ ]:
### Reloading the data 

adata = sc.read_h5ad(PATH+"/Results/Revision_2/Astrocyte_temp_processed.h5ad")
print(adata)
print(adata.X[:10,:10])

## Plotting the UMAP

In [ ]:
mpl.rcParams['svg.fonttype'] = 'none'
sc.pl.embedding(
    adata,
    basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
    color=['leiden_harmony',"region_layer"],
    frameon=False,
    ncols=4,
    size=20,
    # legend_loc="on data",
    use_raw = False,
    save=f"_Astrocyte_CT.svg"
    )

In [ ]:
### Given some known cell type markers, annotate the clusters
mpl.rcParams['svg.fonttype'] = 'none'
# adata = adata.raw.to_adata()
adata.raw = adata.copy()
print(adata.X[:10,:10])

sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)

# Ensure region_layer is categorical
adata.obs['region_layer'] = adata.obs['region_layer'].astype('category')

layer_colors = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Limbic": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    # "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
}

# Map colors to category order (critical — must match adata.obs['region_layer'].cat.categories)
adata.uns['region_layer_colors'] = [
    layer_colors[cat] for cat in adata.obs['region_layer'].cat.categories
]

sc.pl.embedding(
    adata,
    basis='umap_harmony',
    color=["region_layer", "AQP4", 'GFAP', "TNC", "MRVI1", "GRIA1"],
    frameon=False,
    ncols=3,
    size=20,
    use_raw=False,
    save=f"_Astrocyte_CT.svg"
)


## Calculate the region-pairwise distance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

In [ ]:
# ── 1. Extract metadata ────────────────────────────────────────────────────────
meta = adata.obs.copy()

# ── 2. Count table: clusters × brain regions ──────────────────────────────────
counts = (meta.groupby(["leiden_harmony", "region_name"])
              .size()
              .reset_index(name="freq"))
counts.columns = ["cluster", "brain_region", "freq"]

counts["brain_region"] = pd.Categorical(counts["brain_region"],
                                         categories=counts["brain_region"].unique())
counts["cluster"] = counts["cluster"].astype("category")

# ── 3. Wide matrix → proportions ──────────────────────────────────────────────
comp_matrix = counts.pivot_table(index="brain_region", columns="cluster",
                                  values="freq", fill_value=0)
comp_matrix = comp_matrix.div(comp_matrix.sum(axis=1), axis=0)  # row proportions

# ── 4. Hellinger distance ─────────────────────────────────────────────────────
hellinger = np.sqrt(comp_matrix.values)
dist_mat = squareform(pdist(hellinger, metric="euclidean"))
dist_matrix = pd.DataFrame(dist_mat,
                            index=comp_matrix.index,
                            columns=comp_matrix.index)


In [ ]:
# ── 5. Region color mapping ───────────────────────────────────────────────────
layer_colors = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Limbic": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    # "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
}

df_colors = meta[["region_name", "region_layer"]].drop_duplicates()
df_colors["region_color"] = df_colors["region_layer"].map(layer_colors)
region_color = df_colors.set_index("region_name")["region_color"].to_dict()

# ── 6. Row color bar (left annotation) ───────────────────────────────────────
row_colors = pd.Series([region_color.get(r, "#ffffff")
                         for r in dist_matrix.index],
                        index=dist_matrix.index,
                        name="Region")

# ── 7. Clustermap (equivalent to ComplexHeatmap) ─────────────────────────────
mpl.rcParams['svg.fonttype'] = 'none'
np.random.seed(42)

cmap = plt.get_cmap("magma_r")   # rev(magma(100))

g = sns.clustermap(
    dist_matrix,
    # metric="precomputed",       # distance matrix is already computed
    method="average",
    cmap=cmap,
    row_colors=row_colors,
    col_colors=row_colors,
    figsize=(14, 14),
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"label": "Distance"},
)

g.fig.suptitle("Pairwise Distances Between Regions", y=1.01)
g.ax_heatmap.grid(False)
# ── 8. Save ───────────────────────────────────────────────────────────────────
g.savefig("./Results/Revision_2/Figures/Astrocyte_composition_distance_heatmap.pdf",
          bbox_inches="tight")
plt.show()